In [11]:
"""
Compare live flow features (from your Kali -> Lubuntu test) with the CICIDS2017
training data to spot unit or extractor mismatches.

Reading the output:
  orders_off = how many powers of ten the live value is away from the 2017
               median of the TRUE class (about +-3 or +-6 usually means a unit
               problem such as seconds or milliseconds instead of microseconds)
  in_5_95    = whether the live value sits inside the 2017 5th-95th percentile
               range of the true class
"""
import re
import numpy as np
import pandas as pd
import joblib


In [12]:

# ---------------- CONFIG ----------------
LIVE_CSV = r'slowloristest.csv'     # csv written by your live flow extractor
TRUE_CLASS = 'DoS'               # Brute Force | DDoS | DoS | Normal Traffic | Port Scan
ROWS = None                      # None = median of all rows in the file; [0] = first row only; [3, 7] = those rows
PATH_2017 = r'D:\Academic\ML\ModelTraining\Preprocessing\cicids2017_preprocessed.csv'
MODEL = r'D:\Academic\ML\ModelTraining\newOutput\ensemble_model.pkl'
SCALER = r'D:\Academic\ML\ModelTraining\newOutput\scaler.pkl'
ENCODER = r'D:\Academic\ML\ModelTraining\newOutput\label_encoder.pkl'


In [13]:
KEY = ['Flow Duration', 'Flow IAT Mean', 'Flow Packets/s',
       'Fwd Header Length', 'Init_Win_bytes_forward']
# ----------------------------------------

FEATURES = [
    'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets',
    'Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min',
    'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max',
    'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
    'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max',
    'Flow IAT Min', 'Fwd IAT Std', 'Fwd IAT Min', 'Bwd IAT Total',
    'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length',
    'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length',
    'Packet Length Mean', 'Packet Length Std', 'FIN Flag Count',
    'PSH Flag Count', 'ACK Flag Count', 'Init_Win_bytes_forward',
    'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward',
    'Active Mean', 'Active Max', 'Active Min', 'Idle Mean',
]
ABBREV = {
    'Tot Fwd Pkts': 'Total Fwd Packets', 'Tot Bwd Pkts': 'Total Backward Packets',
    'TotLen Fwd Pkts': 'Total Length of Fwd Packets',
    'Fwd Pkt Len Max': 'Fwd Packet Length Max', 'Fwd Pkt Len Min': 'Fwd Packet Length Min',
    'Fwd Pkt Len Mean': 'Fwd Packet Length Mean', 'Fwd Pkt Len Std': 'Fwd Packet Length Std',
    'Bwd Pkt Len Max': 'Bwd Packet Length Max', 'Bwd Pkt Len Mean': 'Bwd Packet Length Mean',
    'Bwd Pkt Len Std': 'Bwd Packet Length Std', 'Flow Byts/s': 'Flow Bytes/s',
    'Flow Pkts/s': 'Flow Packets/s', 'Bwd IAT Tot': 'Bwd IAT Total',
    'Fwd Header Len': 'Fwd Header Length', 'Bwd Header Len': 'Bwd Header Length',
    'Bwd Pkts/s': 'Bwd Packets/s', 'Pkt Len Min': 'Min Packet Length',
    'Pkt Len Max': 'Max Packet Length', 'Pkt Len Mean': 'Packet Length Mean',
    'Pkt Len Std': 'Packet Length Std', 'FIN Flag Cnt': 'FIN Flag Count',
    'PSH Flag Cnt': 'PSH Flag Count', 'ACK Flag Cnt': 'ACK Flag Count',
    'Init Fwd Win Byts': 'Init_Win_bytes_forward',
    'Init Bwd Win Byts': 'Init_Win_bytes_backward',
    'Fwd Act Data Pkts': 'act_data_pkt_fwd', 'Fwd Seg Size Min': 'min_seg_size_forward',
}


In [14]:
def norm(s):
    return re.sub(r'[^a-z0-9]', '', str(s).lower())


# Match live column names ignoring case, spaces, underscores and slashes,
# so "Flow Duration", "flow_duration" and "flow_byts_s" all resolve.
lookup = {norm(f): f for f in FEATURES}
lookup.update({norm(a): f for a, f in ABBREV.items()})

live = pd.read_csv(LIVE_CSV)
live = live.rename(columns={c: lookup[norm(c)] for c in live.columns if norm(c) in lookup})
missing = [f for f in FEATURES if f not in live.columns]
if missing:
    print(f'WARNING - live file has no column for: {missing}')
    print(f'Live columns found: {list(live.columns)}\n')
feats = [f for f in FEATURES if f in live.columns]

live_num = live[feats].apply(pd.to_numeric, errors='coerce').replace([np.inf, -np.inf], np.nan)
sel = live_num.iloc[ROWS] if ROWS is not None else live_num
live_val = sel.median()
print(f'Live rows used: {len(sel)}   true class: {TRUE_CLASS}\n')

d17 = pd.read_csv(PATH_2017, usecols=FEATURES + ['Attack Type'])
cls = d17[d17['Attack Type'] == TRUE_CLASS][feats]
nrm = d17[d17['Attack Type'] == 'Normal Traffic'][feats]
p05, p95 = cls.quantile(0.05), cls.quantile(0.95)

t = pd.DataFrame({
    'live': live_val,
    f'2017 {TRUE_CLASS} median': cls.median(),
    '2017 Normal median': nrm.median(),
})
t['orders_off'] = np.log10(1 + t['live'].abs()) - np.log10(1 + t[f'2017 {TRUE_CLASS} median'].abs())
t['in_5_95'] = (live_val >= p05) & (live_val <= p95)

fmt = lambda v: f'{v:.4g}'
key = [k for k in KEY if k in t.index]
print('KEY FEATURES')
print(t.loc[key].to_string(float_format=fmt))

others = t.drop(index=key)
flagged = others[(others['orders_off'].abs() >= 2) | (~others['in_5_95'])]
print(f'\nOTHER FEATURES outside the 2017 5-95% range or 2+ orders of magnitude off ({len(flagged)} of {len(others)}):')
print(flagged.sort_values('orders_off', key=abs, ascending=False).to_string(float_format=fmt))


Live rows used: 1615   true class: DoS

KEY FEATURES
                            live  2017 DoS median  2017 Normal median  orders_off  in_5_95
Flow Duration          1.121e+07        8.588e+07            4.02e+04     -0.8841     True
Flow IAT Mean           1.89e+06        7.169e+06           1.606e+04      -0.579     True
Flow Packets/s            0.5532           0.1503                  85      0.1304     True
Fwd Header Length            208              200                  64     0.01695     True
Init_Win_bytes_forward       251              251                 122           0     True

OTHER FEATURES outside the 2017 5-95% range or 2+ orders of magnitude off (5 of 35):
                       live  2017 DoS median  2017 Normal median  orders_off  in_5_95
Fwd IAT Min            2331                2                   3       2.891     True
Fwd Packet Length Min    52                0                   6       1.724    False
Min Packet Length        52                0             

In [15]:

# Optional: what the saved model says about the same input
try:
    model, scaler, le = joblib.load(MODEL), joblib.load(SCALER), joblib.load(ENCODER)
    if len(feats) == len(FEATURES):
        x = scaler.transform(live_val[FEATURES].fillna(0).values.reshape(1, -1))
        proba = model.predict_proba(x)[0]
        print('\nModel probabilities for this input:')
        for c, p in sorted(zip(le.classes_, proba), key=lambda z: -z[1]):
            print(f'  {c}: {p:.4f}')
    else:
        print('\nSkipped model probabilities - live file is missing some features.')
except Exception as e:
    print(f'\nSkipped model probabilities: {e}')


Model probabilities for this input:
  Normal Traffic: 0.9848
  DoS: 0.0077
  DDoS: 0.0050
  Port Scan: 0.0025
  Brute Force: 0.0000


In [16]:
print('\nMin length values in live flows (top 8):')
print(live_num['Min Packet Length'].value_counts().head(8).to_string())

model, scaler, le = joblib.load(MODEL), joblib.load(SCALER), joblib.load(ENCODER)
P = model.predict_proba(scaler.transform(live_num[FEATURES].fillna(0).values))
ni = list(le.classes_).index('Normal Traffic')
print('\nPer-flow predictions:')
print(pd.Series(le.classes_[P.argmax(axis=1)]).value_counts().to_string())
print(pd.Series(P[:, ni]).describe(percentiles=[.05, .25, .5, .75, .95]).round(4).to_string())
for t in (0.5, 0.9, 0.99):
    print(f'flows with P(Normal) < {t}: {(P[:, ni] < t).mean():.3f}')


Min length values in live flows (top 8):
Min Packet Length
52     756
40     732
60      40
73      17
220      9
165      9
69       9
49       9

Per-flow predictions:
Normal Traffic    1615
count    1615.0000
mean        0.9639
std         0.0439
min         0.5390
5%          0.9205
25%         0.9561
50%         0.9750
75%         0.9849
95%         0.9925
max         0.9992
flows with P(Normal) < 0.5: 0.000
flows with P(Normal) < 0.9: 0.040
flows with P(Normal) < 0.99: 0.913


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import RobustScaler

LENGTH_FEATS = ['Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min',
                'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max',
                'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s',
                'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std']
MIN_IAT = ['Flow IAT Min', 'Fwd IAT Min', 'Bwd IAT Min']

sample = pd.concat([g.sample(min(len(g), 30000), random_state=42) for _, g in d17.groupby('Attack Type')])
for name, drop in [('A: without length features', LENGTH_FEATS),
                   ('B: without length features and min-IAT features', LENGTH_FEATS + MIN_IAT)]:
    keep = [f for f in FEATURES if f not in drop]
    sc2 = RobustScaler().fit(sample[keep].values)
    rf2 = RandomForestClassifier(n_estimators=100, max_depth=20, class_weight='balanced',
                                 n_jobs=-1, random_state=42).fit(sc2.transform(sample[keep].values),
                                                                  sample['Attack Type'])
    P2 = rf2.predict_proba(sc2.transform(live_num[keep].fillna(0).values))
    ni2 = list(rf2.classes_).index('Normal Traffic')
    print(f'\n{name} ({len(keep)} features) - per-flow predictions on live rows:')
    print(pd.Series(rf2.classes_[P2.argmax(axis=1)]).value_counts().to_string())
    print(f'median P(Normal): {np.median(P2[:, ni2]):.4f}   flows with P(Normal) < 0.9: {(P2[:, ni2] < 0.9).mean():.3f}')


A: without length features (27 features) - per-flow predictions on live rows:
Normal Traffic    1610
DoS                  3
Brute Force          2
median P(Normal): 0.7900   flows with P(Normal) < 0.9: 0.934

B: without length features and min-IAT features (24 features) - per-flow predictions on live rows:
Normal Traffic    1610
DoS                  4
Brute Force          1
median P(Normal): 0.7500   flows with P(Normal) < 0.9: 0.966


In [18]:
print((d17['Init_Win_bytes_backward'] == -1).groupby(d17['Attack Type']).mean().round(3))
print((d17['Init_Win_bytes_forward'] == -1).groupby(d17['Attack Type']).mean().round(3))
print('live backward == 0:', (live_num['Init_Win_bytes_backward'] == 0).mean())

Attack Type
Brute Force       0.008
DDoS              0.363
DoS               0.085
Normal Traffic    0.550
Port Scan         0.001
Name: Init_Win_bytes_backward, dtype: float64
Attack Type
Brute Force       0.000
DDoS              0.000
DoS               0.000
Normal Traffic    0.435
Port Scan         0.000
Name: Init_Win_bytes_forward, dtype: float64
live backward == 0: 0.12198142414860681


In [19]:
z = pd.DataFrame({
    'live zero': (live_num[feats] == 0).mean(),
    '2017 class zero': (cls == 0).mean(),
    'live neg': (live_num[feats] < 0).mean(),
    '2017 class neg': (cls < 0).mean(),
})
z['zero gap'] = (z['live zero'] - z['2017 class zero']).abs()
z['neg gap'] = (z['live neg'] - z['2017 class neg']).abs()
print(z[(z['zero gap'] > 0.3) | (z['neg gap'] > 0.1)].round(3).to_string())

                       live zero  2017 class zero  live neg  2017 class neg  zero gap  neg gap
Fwd Packet Length Min      0.001            0.951       0.0             0.0     0.950      0.0
Min Packet Length          0.000            0.997       0.0             0.0     0.997      0.0
PSH Flag Count             0.059            0.845       0.0             0.0     0.786      0.0
ACK Flag Count             0.037            0.458       0.0             0.0     0.421      0.0


In [20]:
def flag_profile(d):
    pk = d['Total Fwd Packets'] + d['Total Backward Packets']
    has_data = d['act_data_pkt_fwd'] > 0
    return pd.Series({
        'rows': len(d),
        'median packets per flow': pk.median(),
        'PSH==0 among flows with fwd data packets': (d.loc[has_data, 'PSH Flag Count'] == 0).mean(),
        'ACK==0 among flows with 3+ packets': (d.loc[pk >= 3, 'ACK Flag Count'] == 0).mean(),
        'median PSH / packets': (d['PSH Flag Count'] / pk).median(),
        'median ACK / packets': (d['ACK Flag Count'] / pk).median(),
    })

print(pd.DataFrame({f'2017 {TRUE_CLASS}': flag_profile(cls),
                    '2017 Normal': flag_profile(nrm),
                    'live': flag_profile(live_num[feats])}).round(3).to_string())

                                            2017 DoS  2017 Normal      live
rows                                      193647.000  2094458.000  1615.000
median packets per flow                       13.000        4.000     7.000
PSH==0 among flows with fwd data packets       0.857        0.622     0.032
ACK==0 among flows with 3+ packets             0.468        0.852     0.012
median PSH / packets                           0.000        0.000     0.375
median ACK / packets                           0.062        0.000     0.875
